# ReAct vs Native Tool Calling: Side by Side

**Prerequisite:** finish notebooks 01–03 first.

You have now built agents two ways:
- **Native tool calling** (notebook 01): the model emits structured `tool_calls` JSON, the API handles the format
- **ReAct** (notebooks 02 and 04): the model writes plain text following a template, we parse with regex

This notebook runs the **same questions** through both approaches and measures:
- Number of LLM calls
- Total tokens consumed
- Whether parallel tool calls happened
- Robustness (did the format break?)

No new concepts. Just measurement.

In [ ]:
!uv pip install -q openai tiktoken python-dotenv requests

In [ ]:
import os, re, time, requests
from dotenv import load_dotenv
from openai import OpenAI
import tiktoken

load_dotenv()
client = OpenAI()
enc = tiktoken.encoding_for_model("gpt-4o-mini")

# ── Shared tools (plain functions) ──
def add(a=None, b=None, expression=None):
    if expression:
        a, b = [int(x.strip()) for x in expression.split(",")]
    return str(int(a) + int(b))

def multiply(a=None, b=None, expression=None):
    if expression:
        a, b = [int(x.strip()) for x in expression.split(",")]
    return str(int(a) * int(b))

def web_search(query: str) -> str:
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        return f"[FAKE SEARCH] pretend results for: {query}"
    r = requests.post("https://api.tavily.com/search",
                      json={"api_key": api_key, "query": query, "max_results": 3}, timeout=15)
    r.raise_for_status()
    return "\n".join(f"- {i['title']}: {i['content'][:200]}" for i in r.json().get("results", []))

TOOL_FNS = {"add": add, "multiply": multiply, "web_search": web_search}

# ── ReAct agent ──
REACT_PROMPT = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
{scratchpad}"""

react_tools_text = "\n".join([
    "add: Add two integers. Input: two integers separated by a comma, e.g. '17, 25'",
    "multiply: Multiply two integers. Input: two integers separated by a comma, e.g. '6, 7'",
    "web_search: Search the web. Input: a plain search query string",
])
react_tool_names = "add, multiply, web_search"

def parse_react(text):
    final = re.search(r"Final Answer:\s*(.*)", text, re.DOTALL)
    if final:
        return {"final_answer": final.group(1).strip()}
    action = re.search(r"Action:\s*(.*?)(?:\n|$)", text)
    action_input = re.search(r"Action Input:\s*(.*)", text, re.DOTALL)
    if action and action_input:
        return {"action": action.group(1).strip(), "action_input": action_input.group(1).strip()}
    raise ValueError(f"Parse error:\n{text}")

def run_react(question, max_steps=8):
    metrics = {"llm_calls": 0, "tool_calls": 0, "total_tokens": 0,
               "parallel_calls": 0, "errors": 0, "answer": None}
    scratchpad = ""
    t0 = time.time()
    for step in range(1, max_steps + 1):
        prompt = REACT_PROMPT.format(tools=react_tools_text, tool_names=react_tool_names,
                                     input=question, scratchpad=scratchpad)
        response = client.chat.completions.create(
            model="gpt-4o-mini", temperature=0,
            stop=["\nObservation:", "Observation:"],
            messages=[{"role": "user", "content": prompt}],
        )
        metrics["llm_calls"] += 1
        metrics["total_tokens"] += response.usage.total_tokens
        text = response.choices[0].message.content
        try:
            parsed = parse_react(text)
        except ValueError:
            metrics["errors"] += 1
            break
        if "final_answer" in parsed:
            metrics["answer"] = parsed["final_answer"]
            break
        action, action_input = parsed["action"], parsed["action_input"]
        metrics["tool_calls"] += 1
        try:
            observation = TOOL_FNS[action](expression=action_input)
        except Exception as e:
            observation = f"Error: {e}"
            metrics["errors"] += 1
        scratchpad += f"{text}\nObservation: {observation}\n"
    metrics["wall_time"] = round(time.time() - t0, 2)
    return metrics

# ── Native tool calling agent ──
NATIVE_TOOLS = [
    {"type": "function", "function": {
        "name": "add", "description": "Add two integers and return the sum.",
        "parameters": {"type": "object", "properties": {
            "a": {"type": "integer", "description": "First integer"},
            "b": {"type": "integer", "description": "Second integer"}
        }, "required": ["a", "b"]}}},
    {"type": "function", "function": {
        "name": "multiply", "description": "Multiply two integers and return the product.",
        "parameters": {"type": "object", "properties": {
            "a": {"type": "integer", "description": "First integer"},
            "b": {"type": "integer", "description": "Second integer"}
        }, "required": ["a", "b"]}}},
    {"type": "function", "function": {
        "name": "web_search", "description": "Search the web for current information.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "Search query"}
        }, "required": ["query"]}}},
]

def run_native(question, max_steps=8):
    metrics = {"llm_calls": 0, "tool_calls": 0, "total_tokens": 0,
               "parallel_calls": 0, "errors": 0, "answer": None}
    messages = [{"role": "user", "content": question}]
    t0 = time.time()
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model="gpt-4o-mini", temperature=0,
            tools=NATIVE_TOOLS, messages=messages,
        )
        metrics["llm_calls"] += 1
        metrics["total_tokens"] += response.usage.total_tokens
        msg = response.choices[0].message
        if not msg.tool_calls:
            metrics["answer"] = msg.content
            messages.append({"role": "assistant", "content": msg.content})
            break
        # Track parallel calls
        if len(msg.tool_calls) > 1:
            metrics["parallel_calls"] += 1
        messages.append({"role": "assistant", "content": msg.content, "tool_calls": [
            {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
            for tc in msg.tool_calls
        ]})
        for tc in msg.tool_calls:
            import json as _json
            args = _json.loads(tc.function.arguments)
            metrics["tool_calls"] += 1
            try:
                result = TOOL_FNS[tc.function.name](**args)
            except Exception as e:
                result = f"Error: {e}"
                metrics["errors"] += 1
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
    metrics["wall_time"] = round(time.time() - t0, 2)
    return metrics

print("Both agents ready.")

## Test 1: Simple sequential task

A question that requires two tool calls in sequence (the second depends on the first).

In [ ]:
q1 = "What is 12 multiplied by 7, then add 100 to that?"

print("Running ReAct...")
react_1 = run_react(q1)
print(f"  Answer: {react_1['answer']}")

print("\nRunning Native...")
native_1 = run_native(q1)
print(f"  Answer: {native_1['answer']}")

## Test 2: Parallel-eligible task

A question where two tool calls are independent and can run in parallel.

In [ ]:
q2 = "What is 5 + 3, and also what is 10 * 4?"

print("Running ReAct...")
react_2 = run_react(q2)
print(f"  Answer: {react_2['answer']}")

print("\nRunning Native...")
native_2 = run_native(q2)
print(f"  Answer: {native_2['answer']}")

## Test 3: Three-step chain

A question requiring three sequential tool calls.

In [ ]:
q3 = "Multiply 6 by 8, then add 50, then add 25 more."

print("Running ReAct...")
react_3 = run_react(q3)
print(f"  Answer: {react_3['answer']}")

print("\nRunning Native...")
native_3 = run_native(q3)
print(f"  Answer: {native_3['answer']}")

## Comparison table

In [ ]:
tests = [
    ("Sequential (2 steps)", react_1, native_1),
    ("Parallel-eligible", react_2, native_2),
    ("3-step chain", react_3, native_3),
]

header = f"{'Test':<24} {'Metric':<20} {'ReAct':<12} {'Native':<12} {'Winner':<10}"
print(header)
print("─" * len(header))

for name, react, native in tests:
    metrics = ["llm_calls", "tool_calls", "total_tokens", "parallel_calls", "wall_time"]
    labels = ["LLM calls", "Tool calls", "Total tokens", "Parallel batches", "Wall time (s)"]
    for metric, label in zip(metrics, labels):
        r_val = react[metric]
        n_val = native[metric]
        if metric == "parallel_calls":
            winner = "native" if n_val > r_val else ("tie" if n_val == r_val else "react")
        elif metric == "wall_time":
            winner = "react" if r_val < n_val else ("tie" if r_val == n_val else "native")
        else:
            winner = "react" if r_val < n_val else ("tie" if r_val == n_val else "native")
        row_name = name if label == "LLM calls" else ""
        print(f"{row_name:<24} {label:<20} {r_val:<12} {n_val:<12} {winner:<10}")
    print()

## Key observations

Read the table above. The patterns you should see:

In [ ]:
print("1. TOKENS")
print(f"   ReAct used {react_1['total_tokens']} tokens for the sequential task.")
print(f"   Native used {native_1['total_tokens']} tokens for the same task.")
print(f"   Difference: {react_1['total_tokens'] - native_1['total_tokens']:+d} tokens")
print(f"   ReAct is more expensive because it resends the entire prompt+scratchpad each step.")
print()

print("2. PARALLEL TOOL CALLS")
print(f"   On the parallel-eligible question:")
print(f"   ReAct parallel batches: {react_2['parallel_calls']} (always 0 — one tool per step)")
print(f"   Native parallel batches: {native_2['parallel_calls']}")
if native_2['parallel_calls'] > 0:
    print(f"   Native ran {native_2['tool_calls']} tools in {native_2['parallel_calls']} batch(es) in a single LLM response.")
else:
    print(f"   (Model chose to run sequentially this time — parallel is not guaranteed.)")
print()

print("3. LLM CALLS")
print(f"   ReAct always needs N+1 LLM calls for N tool uses (one per tool + final answer).")
print(f"   Native can batch parallel tools, potentially needing fewer calls.")

## When to use which

| | Native tool calling | ReAct |
|---|---|---|
| **Use in production** | Yes | Rarely |
| **Works on any model** | No (needs tool calling support) | Yes (any text model) |
| **Parallel tool calls** | Yes | No |
| **Token efficiency** | Better (structured messages, no prompt resend) | Worse (full prompt + scratchpad every step) |
| **Format robustness** | High (API enforces JSON schema) | Lower (model can break the text format) |
| **Transparency** | Tool calls are structured data | Everything is readable plain text |
| **Teaching value** | Shows modern API capabilities | Shows there is no magic underneath |

## The evolution

```text
ReAct (2022)                     Native tool calling (2023+)         Modern frameworks
─────────────                    ──────────────────────────          ─────────────────
Text template + regex parsing    Structured JSON tool calls          LangGraph, CrewAI, etc.
One tool per turn                Parallel tool calls                 Multi-agent orchestration
Manual prompt engineering        API-managed format                  State machines, memory, planning
Works on any LLM                 Requires model support              Built on native tool calling
```

ReAct proved the concept. Native tool calling productionized it. Frameworks add orchestration on top. But the core loop is always the same: **call the model → run the tool → feed the result back → repeat.**

You now understand all three layers.

## What's next

You have completed the core agent curriculum:

| Notebook | What you learned |
|---|---|
| 01 | What tool calling is — JSON in, JSON out |
| 02 | ReAct from zero — no framework, ~50 lines |
| 03 | Failure modes — why each component exists |
| 04 | The ReAct pattern with LangChain — the original agent prompt |
| 05 | What the model sees — token costs, prompt growth |
| 06 | ReAct vs native — tradeoffs with real numbers |

From here, you can explore:
- **Multi-agent systems** — multiple agents collaborating or delegating
- **Memory and state** — agents that remember across conversations  
- **Planning** — agents that decompose tasks before acting
- **Evaluation** — measuring agent quality systematically